# 예제 03. 전처리 파이프라인
빅데이터프로그래밍 · 3주차

## 목표
- 필요 없는 열을 정리한다
- 데이터 순서를 섞고 학습/검증으로 나눈다
- 값의 범위를 맞춘다 (정규화 · 표준화)

순서가 중요합니다. **나눈 다음에 범위를 맞춥니다.**


In [ ]:
csv = """name,gender,department,study_hours,attendance,midterm,final
김통계,남,통계학과,12.5,95%,88,92
이확률,여,통계학과,8.0,88%,92,85
박회귀,남,컴퓨터공학과,,72%,79,68
최추정,여,통계학과,15.0,100%,95,98
정검정,남,경제학과,4.5,61%,61,
한분산,여,컴퓨터공학과,10.0,90%,84,88
오평균,남,경제학과,6.5,,70,74
서표본,여,통계학과,13.0,97%,100,96
남표준,남,컴퓨터공학과,9.5,85%,66,71
윤편차,여,경제학과,,80%,82,79
"""

with open("students.csv", "w", encoding="utf-8") as f:
    f.write(csv)

print("students.csv 생성 완료")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("students.csv")

# 앞 예제의 처리를 한 번에
df["study_hours"] = df["study_hours"].fillna(df["study_hours"].mean())
df["final"] = df["final"].fillna(df["final"].mean())
df["attendance"] = (df["attendance"].str.replace("%", "", regex=False).astype(float))
df["attendance"] = df["attendance"].fillna(df["attendance"].median())
df["gender"] = df["gender"].map({"남": 0, "여": 1})
df = pd.get_dummies(df, columns=["department"], dtype=int)

df.head(3)


## 1. 필요 없는 열 삭제
이름은 예측에 쓸 수 없는 식별자입니다.


In [ ]:
df = df.drop(columns=["name"])
print(df.columns.tolist())


## 2. 목표값 만들기
`final` 이 80점 이상이면 1, 아니면 0 — 이진 분류 문제로 만듭니다.


In [ ]:
df["pass"] = (df["final"] >= 80).astype(int)

X = df.drop(columns=["pass", "final"])   # 입력
y = df["pass"]                            # 정답

print("X shape:", X.shape, "/ y shape:", y.shape)
print(y.value_counts().to_dict())


## 3. 순서 섞기
데이터가 학과순·점수순으로 정렬돼 있으면 앞부분만 학습하게 됩니다.
`random_state` 를 고정하면 매번 같은 결과가 나옵니다.


In [ ]:
shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
shuffled.head(3)


## 4. 학습 / 검증 분할


In [ ]:
X = shuffled.drop(columns=["pass", "final"])
y = shuffled["pass"]

n_train = int(len(X) * 0.7)

X_train, X_val = X.iloc[:n_train], X.iloc[n_train:]
y_train, y_val = y.iloc[:n_train], y.iloc[n_train:]

print("학습:", X_train.shape, "검증:", X_val.shape)


## 5. 값의 범위 맞추기

| 방법 | 식 | 결과 범위 |
| --- | --- | --- |
| 정규화 min-max | (x − min) / (max − min) | 0 ~ 1 |
| 표준화 z-score | (x − 평균) / 표준편차 | 평균 0, 표준편차 1 |

**기준은 항상 학습 데이터입니다.** 검증 데이터의 평균을 쓰면 정보가 새어 들어갑니다.


In [ ]:
num_cols = ["study_hours", "attendance", "midterm"]

mean = X_train[num_cols].mean()      # 학습 데이터 기준
std = X_train[num_cols].std()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()

X_train_scaled[num_cols] = (X_train[num_cols] - mean) / std
X_val_scaled[num_cols] = (X_val[num_cols] - mean) / std   # 같은 기준을 적용

print(X_train_scaled[num_cols].round(2))
print("\n학습셋 평균:", X_train_scaled[num_cols].mean().round(6).to_dict())


## 6. 전처리 요약 확인


In [ ]:
print("입력 열 :", X_train_scaled.columns.tolist())
print("학습 shape:", X_train_scaled.shape)
print("검증 shape:", X_val_scaled.shape)
print("결측 개수 :", X_train_scaled.isnull().sum().sum())
print("자료형    :", set(X_train_scaled.dtypes.astype(str)))


## 직접 해보기
1. 표준화 대신 min-max 정규화로 바꿔 보세요.
2. 분할 비율을 8:2 로 바꾸면 각 shape이 어떻게 되나요?


In [ ]:
# 여기에 작성하세요
